# STEP 1 

## 🚊 STEP 1-1 코드 전체 (정거장 500m Buffer 생성까지)
- 정거장 500m 버퍼

- 2022년 도로망 좌표계 변환

- 이후 Spatial Join 가능 상태

In [2]:
# ============================================================
# STEP 1. 정거장 500m 접근성 분석 준비 코드 (대전 2호선)
# ============================================================

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ------------------------------------------------------------
# 1) 정거장 좌표(CSV) 불러오기
# ------------------------------------------------------------
station_path = "C:/Users/dhso2/Downloads/2호선_위_경도 (1).csv"
stations = pd.read_csv(station_path)

print("정거장 CSV 컬럼:")
print(stations.columns)
print(stations.head())


# ------------------------------------------------------------
# 2) GeoDataFrame으로 변환 (EPSG:4326 = 위경도)
# ------------------------------------------------------------
gdf_st = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations["경도"], stations["위도"]),
    crs="EPSG:4326"    # WGS84
)

print("\n[정거장 GeoDataFrame 생성 완료]")


# ------------------------------------------------------------
# 3) 거리 기반 버퍼를 만들기 위해 국가좌표계(EPSG:5179)로 변환
# ------------------------------------------------------------
gdf_st_5179 = gdf_st.to_crs(epsg=5179)

print("\n좌표계 변환 완료 → EPSG:5179 (미터 단위)")


# ------------------------------------------------------------
# 4) 각 정거장별 500m Buffer 생성
# ------------------------------------------------------------
gdf_st_5179["buffer_500m"] = gdf_st_5179.geometry.buffer(500)

# 버퍼만 따로 GeoDataFrame 생성 (나중에 지도 시각화 편함)
buffer_gdf = gdf_st_5179[["정거장ID", "정거장명", "buffer_500m"]].copy()
buffer_gdf = buffer_gdf.set_geometry("buffer_500m")

print("\n[500m 버퍼 생성 완료]")
print(buffer_gdf.head())


# ------------------------------------------------------------
# 5) 2022년 도로망 데이터 불러오기
# ------------------------------------------------------------
road_path = "C:/Users/dhso2/Downloads/32.대전광역시_level5_5_link_주요도로망_형상_2022.geojson"

roads = gpd.read_file(road_path)

print("\n도로망 컬럼:")
print(roads.columns)
print(roads.head())


# ------------------------------------------------------------
# 6) 도로망도 좌표계를 5179로 변환
# ------------------------------------------------------------
roads_5179 = roads.to_crs(epsg=5179)

print("\n도로망 좌표계 변환 완료 → EPSG:5179")


# ------------------------------------------------------------
# 7) 확인
# ------------------------------------------------------------
print("\n========================================")
print("STEP 1 준비 완료!")
print("이제 각 정거장 500m 안에 들어오는 도로 링크를 계산할 수 있습니다.")
print("다음 단계: 도로–정거장 buffer spatial join")
print("========================================")


정거장 CSV 컬럼:
Index(['정거장ID', '정거장명', '위도', '경도', '_PARCEL_AD', '_ROAD_AD'], dtype='object')
   정거장ID   정거장명         위도          경도            _PARCEL_AD  \
0    201   서대전역  36.320142  127.404468      대전광역시 중구 오류동 195   
1    202  서대전 4  36.322083  127.411119      대전광역시 중구 오류동 195   
2    203     대사  36.319332  127.416331  대전광역시 중구 대사동 248-225   
3    204     대흥  36.318118  127.428438    대전광역시 중구 대사동 77-51   
4    205     인동  36.322842  127.437252       대전광역시 동구 인동 352   

             _ROAD_AD  
0   대전광역시 중구 계백로 1663  
1   대전광역시 중구 계백로 1719  
2    대전광역시 중구 계룡로 937  
3  대전광역시 중구 충무로 104-1  
4    대전광역시 동구 대전로 702  

[정거장 GeoDataFrame 생성 완료]

좌표계 변환 완료 → EPSG:5179 (미터 단위)

 버퍼 생성 완료]
   정거장ID   정거장명                                        buffer_500m
0    201   서대전역  POLYGON ((991924.861 1813647.046, 991922.453 1...
1    202  서대전 4  POLYGON ((992522.066 1813861.774, 992519.658 1...
2    203     대사  POLYGON ((992989.63 1813556.214, 992987.222 18...
3    204     대흥  POLYGON ((994076.289 18134

## 🚊 Step 1-2 코드(Spatial Join):

In [3]:
# ------------------------------------------------------------
# STEP 1-2: 정거장 500m 버퍼 안의 도로 추출
# ------------------------------------------------------------

# buffer = polygon, roads = linestring → spatial join
join = gpd.sjoin(
    roads_5179,
    buffer_gdf,
    predicate="intersects",
    how="inner"
)

print(join.head())
print(join[["정거장명", "k_link_id"]].head())

print("\n정거장별 도로 링크 개수:")
print(join.groupby("정거장명")["k_link_id"].count())


    k_link_id  fnode_id  tnode_id road_name road_no  road_rank  link_type  \
4   1026322.0  278075.0  277811.0       배재로       0      104.0    32768.0   
5   1026323.0  277828.0  277911.0       구봉로       0      104.0    32768.0   
6   1026324.0  277911.0  277828.0       구봉로       0      104.0    32768.0   
13  1026335.0  277924.0  278113.0     구봉산북로       0      104.0    32768.0   
14  1026336.0  277907.0  277965.0      관저동로       0      104.0    32768.0   

    lane road_info  sido_id  sigungu_id      emd_id  k_length  \
4    4.0      None  25000.0     25030.0  25030530.0     1.037   
5    6.0      None  25000.0     25040.0  25040680.0     1.082   
6    6.0      None  25000.0     25030.0  25030730.0     1.082   
13   4.0      None  25000.0     25030.0  25030720.0     0.615   
14   6.0      None  25000.0     25030.0  25030730.0     0.658   

                                             geometry  index_right  정거장ID  \
4   LINESTRING (988189.303 1812594.191, 988188.617...           35   

In [5]:
import pandas as pd

cong = pd.read_csv("C:/Users/dhso2/Downloads/22.대전광역시_평일_혼잡강도.csv")

print("혼잡강도 CSV 컬럼:")
print(cong.columns)

print("\n상위 5개 행:")
print(cong.head())


혼잡강도 CSV 컬럼:
Index(['기준연도', '5.5 LINK ID', 'ITS LINK ID', '도로등급', '도로명', '권역', '연장(km)',
       '차선수', '혼잡빈도강도', '혼잡시간강도', '혼잡기대강도'],
      dtype='object')

상위 5개 행:
   기준연도  5.5 LINK ID                       ITS LINK ID    도로등급       도로명  \
0  2019      1026309                        1850000600    고속도로  호남고속도로지선   
1  2019      1026314  1850177000/1850182500/2910000502    고속도로  호남고속도로지선   
2  2019      1026318                        1850058200  특별광역시도      동서대로   
3  2019      1026319             1850055900/1850057600  특별광역시도      동서대로   
4  2019      1026320             1850060700/1850210800  특별광역시도      동서대로   

          권역  연장(km)  차선수  혼잡빈도강도  혼잡시간강도  혼잡기대강도  
0   대전광역시 서구  0.4280    2      19      25      30  
1  대전광역시 유성구  5.9944    2      29      36      29  
2   대전광역시 서구  0.5269    4      24      40      62  
3   대전광역시 서구  0.5058    4      31      54      60  
4   대전광역시 서구  0.4677    3      27      47      65  


## 🚊 STEP 1-3: 정거장 500m Buffer × 혼잡강도 매칭 전체 코드
✔ 매칭 핵심

- 도로망의 링크 ID = k_link_id

- 혼잡강도의 링크 ID = 5.5 LINK ID

즉, 이 두 컬럼을 기준으로 merge 해준다.

In [6]:
# ============================================================
# STEP 1-3. 정거장 500m Buffer × 혼잡강도 매칭
# ============================================================

import pandas as pd
import geopandas as gpd

# ------------------------------------------------------------
# 1) 이미 만든 도로망 Spatial Join 결과 다시 가져오기
#     (join = roads_5179 ⊂ buffer_gdf)
# ------------------------------------------------------------
# join 변수는 이전에 생성한 spatial join 결과라고 가정한다.


# ------------------------------------------------------------
# 2) 혼잡강도 CSV 불러오기
# ------------------------------------------------------------
cong = pd.read_csv("C:/Users/dhso2/Downloads/22.대전광역시_평일_혼잡강도.csv")

# 컬럼명 정리 (공백 포함되어 있으므로 rename 권장)
cong = cong.rename(columns={
    "5.5 LINK ID": "k_link_id",
    "혼잡빈도강도": "freq_idx",
    "혼잡시간강도": "time_idx",
    "혼잡기대강도": "exp_idx"
})

print("혼잡강도 CSV 구조 확인:")
print(cong.head())


# ------------------------------------------------------------
# 3) k_link_id 기준으로 Spatial Join 결과와 merge
# ------------------------------------------------------------
# join에는 도로망(k_link_id) + 정거장명 있음
join_cong = join.merge(
    cong[["k_link_id", "freq_idx", "time_idx", "exp_idx"]],
    on="k_link_id",
    how="left"
)

print("\n정거장 500m 내 도로 + 혼잡강도 매칭 결과:")
print(join_cong.head())


# ------------------------------------------------------------
# 4) 정거장별 평균 혼잡도 계산
# ------------------------------------------------------------
station_cong = join_cong.groupby("정거장명")[["freq_idx", "time_idx", "exp_idx"]].mean().reset_index()

print("\n정거장별 평균 혼잡도:")
print(station_cong)


# ------------------------------------------------------------
# 5) 정거장별 혼잡도 순위 (Top 10)
# ------------------------------------------------------------
station_rank = station_cong.sort_values("time_idx", ascending=False)
print("\n🚨 정거장별 혼잡시간강도 Top 10:")
print(station_rank.head(10))


혼잡강도 CSV 구조 확인:
   기준연도  k_link_id                       ITS LINK ID    도로등급       도로명  \
0  2019    1026309                        1850000600    고속도로  호남고속도로지선   
1  2019    1026314  1850177000/1850182500/2910000502    고속도로  호남고속도로지선   
2  2019    1026318                        1850058200  특별광역시도      동서대로   
3  2019    1026319             1850055900/1850057600  특별광역시도      동서대로   
4  2019    1026320             1850060700/1850210800  특별광역시도      동서대로   

          권역  연장(km)  차선수  freq_idx  time_idx  exp_idx  
0   대전광역시 서구  0.4280    2        19        25       30  
1  대전광역시 유성구  5.9944    2        29        36       29  
2   대전광역시 서구  0.5269    4        24        40       62  
3   대전광역시 서구  0.5058    4        31        54       60  
4   대전광역시 서구  0.4677    3        27        47       65  

정거장 500m 내 도로 + 혼잡강도 매칭 결과:
   k_link_id  fnode_id  tnode_id road_name road_no  road_rank  link_type  \
0  1026322.0  278075.0  277811.0       배재로       0      104.0    32768.0   
1  1026322.0  27

✔ 1) 정거장별 평균 혼잡도

- 혼잡빈도강도 평균(freq_idx)

- 혼잡시간강도 평균(time_idx)

- 혼잡기대강도 평균(exp_idx)

✔ 2) 정거장 혼잡도 랭킹

- 혼잡시간강도 기준 Top 10

- “트램 필요성이 가장 높은 정거장” 목록 자동 생성

In [7]:
spd = pd.read_csv("C:/Users/dhso2/Downloads/20.대전광역시_평일_혼잡시 평균속도.csv")
print(spd.columns)
print(spd.head())


Index(['기준연도', '5.5 LINK ID', 'ITS LINK ID', '도로등급', '도로명', '권역', '연장(km)',
       '차선수', '전일', '1시', '2시', '3시', '4시', '5시', '6시', '7시', '8시', '9시',
       '10시', '11시', '12시', '13시', '14시', '15시', '16시', '17시', '18시', '19시',
       '20시', '21시', '22시', '23시', '24시'],
      dtype='object')
   기준연도  5.5 LINK ID                       ITS LINK ID    도로등급       도로명  \
0  2019      1026309                        1850000600    고속도로  호남고속도로지선   
1  2019      1026314  1850177000/1850182500/2910000502    고속도로  호남고속도로지선   
2  2019      1026318                        1850058200  특별광역시도      동서대로   
3  2019      1026319             1850055900/1850057600  특별광역시도      동서대로   
4  2019      1026320             1850060700/1850210800  특별광역시도      동서대로   

          권역  연장(km)  차선수  전일  1시  ... 15시 16시 17시 18시 19시 20시 21시 22시 23시 24시  
0   대전광역시 서구  0.4280    2  82  80  ...  83  84  81  83  79  83  83  83  81  80  
1  대전광역시 유성구  5.9944    2  79  81  ...  80  82  80  80  70  75  81  81  81  81  
2   대전광역

## 🚊 STEP 1-4: 정거장 500m × 평균속도(혼잡/정상) 매칭 코드
✔ 목적

- 정거장 500m 내부에 포함된 도로들의 “평균속도”를 계산

- 시간대별 속도 모두 활용 가능 (전일, 1시~24시)

- 최종적으로 정거장별 혼잡시/정상시 평균속도 score 산출

In [9]:
# ============================================================
# STEP 1-4. 정거장 500m Buffer × 평균속도(혼잡/정상) 매칭
# ============================================================

import pandas as pd
import geopandas as gpd

# ------------------------------------------------------------
# 1) 혼잡시 평균속도 불러오기
# ------------------------------------------------------------
spd_cong = pd.read_csv("C:/Users/dhso2/Downloads/20.대전광역시_평일_혼잡시 평균속도.csv")

# 컬럼명 통일 (혼잡강도처럼 링크 ID를 k_link_id로 맞춤)
spd_cong = spd_cong.rename(columns={"5.5 LINK ID": "k_link_id"})

print("혼잡시 평균속도 CSV 구조 확인:")
print(spd_cong.head())


# ------------------------------------------------------------
# 2) 정상시 평균속도 불러오기
# ------------------------------------------------------------
spd_norm = pd.read_csv("C:/Users/dhso2/Downloads/21.대전광역시_평일_정상시 평균속도.csv")
spd_norm = spd_norm.rename(columns={"5.5 LINK ID": "k_link_id"})

print("\n정상시 평균속도 CSV 구조 확인:")
print(spd_norm.head())


# ------------------------------------------------------------
# 3) 혼잡시/정상시 평균속도에서 필요한 속도 컬럼만 추출
# ------------------------------------------------------------
speed_cols = ['전일', '1시', '2시', '3시', '4시', '5시', '6시', '7시', '8시', '9시',
              '10시', '11시', '12시', '13시', '14시', '15시', '16시', '17시', '18시',
              '19시', '20시', '21시', '22시', '23시', '24시']

spd_cong_small = spd_cong[['k_link_id'] + speed_cols]
spd_norm_small = spd_norm[['k_link_id'] + speed_cols]


# ------------------------------------------------------------
# 4) Spatial Join 결과(join)에 평균속도 merge
# ------------------------------------------------------------
# join = (도로망 5179) ∩ (정거장 500m buffer)

join_spd = join.merge(
    spd_cong_small,
    on='k_link_id',
    how='left'
).merge(
    spd_norm_small,
    on='k_link_id',
    how='left',
    suffixes=('_cong', '_norm')
)

print("\n정거장 500m 내부 도로 + 평균속도 매칭 결과:")
print(join_spd.head())


# ============================================================
# 🔧 오류 해결: 속도 컬럼 numeric 변환 (mean 적용을 위해 필수)
# ============================================================

# 1) 변환할 컬럼 목록 생성
cong_speed_cols = [col for col in join_spd.columns if col.endswith('_cong')]
norm_speed_cols = [col for col in join_spd.columns if col.endswith('_norm')]

# 2) 모든 속도 컬럼을 숫자(float)로 변환
for col in cong_speed_cols + norm_speed_cols:
    join_spd[col] = pd.to_numeric(join_spd[col], errors='coerce')

print("\n[모든 속도 컬럼 numeric 변환 완료]\n")


# ------------------------------------------------------------
# 5) 정거장별 혼잡시 평균속도 & 정상시 평균속도 계산
# ------------------------------------------------------------
station_cong_speed = join_spd.groupby("정거장명")[cong_speed_cols].mean().reset_index()
station_norm_speed = join_spd.groupby("정거장명")[norm_speed_cols].mean().reset_index()

print("\n🚦 정거장별 혼잡시 평균속도:")
print(station_cong_speed.head())

print("\n🚦 정거장별 정상시 평균속도:")
print(station_norm_speed.head())


# ------------------------------------------------------------
# 6) 정거장별 속도지표 단일 Score 생성 (전일 기준)
# ------------------------------------------------------------
station_cong_speed['혼잡시_전일_평균속도'] = station_cong_speed['전일_cong']
station_norm_speed['정상시_전일_평균속도'] = station_norm_speed['전일_norm']

speed_summary = station_cong_speed[['정거장명', '혼잡시_전일_평균속도']].merge(
    station_norm_speed[['정거장명', '정상시_전일_평균속도']],
    on='정거장명',
    how='left'
)

print("\n🚀 최종 정거장별 속도 Summary:")
print(speed_summary.sort_values("혼잡시_전일_평균속도"))


혼잡시 평균속도 CSV 구조 확인:
   기준연도  k_link_id                       ITS LINK ID    도로등급       도로명  \
0  2019    1026309                        1850000600    고속도로  호남고속도로지선   
1  2019    1026314  1850177000/1850182500/2910000502    고속도로  호남고속도로지선   
2  2019    1026318                        1850058200  특별광역시도      동서대로   
3  2019    1026319             1850055900/1850057600  특별광역시도      동서대로   
4  2019    1026320             1850060700/1850210800  특별광역시도      동서대로   

          권역  연장(km)  차선수  전일  1시  ... 15시 16시 17시 18시 19시 20시 21시 22시 23시 24시  
0   대전광역시 서구  0.4280    2  82  80  ...  83  84  81  83  79  83  83  83  81  80  
1  대전광역시 유성구  5.9944    2  79  81  ...  80  82  80  80  70  75  81  81  81  81  
2   대전광역시 서구  0.5269    4  19  19  ...  19  19  19  20  20  20  20  20  19  19  
3   대전광역시 서구  0.5058    4  15  14  ...  15  15  15  15  15  15  15  15  15  16  
4   대전광역시 서구  0.4677    3  16  18  ...  16  17  16  16  16  16  17  17  16  17  

[5 rows x 33 columns]

정상시 평균속도 CSV 구조 확인:
   기준

## 🚀 STEP 1-5 전체 코드 (정거장별 추정교통량 평균 계산)

In [10]:
df = pd.read_csv("C:/Users/dhso2/Downloads/17.대전광역시_평일_추정교통량.csv")
print(df.columns)
print(df.head())


Index(['기준연도', '5.5 LINK ID', 'ITS LINK ID', '도로등급', '도로명', '권역', '연장(km)',
       '차선수', '전체-평일_전일', '전체-평일_1시',
       ...
       '트럭-평일_15시', '트럭-평일_16시', '트럭-평일_17시', '트럭-평일_18시', '트럭-평일_19시',
       '트럭-평일_20시', '트럭-평일_21시', '트럭-평일_22시', '트럭-평일_23시', '트럭-평일_24시'],
      dtype='object', length=108)
   기준연도  5.5 LINK ID                       ITS LINK ID    도로등급       도로명  \
0  2019      1026307                        1850000300     연결로  호남고속도로지선   
1  2019      1026308                        1850000700     연결로  호남고속도로지선   
2  2019      1026309                        1850000600    고속도로  호남고속도로지선   
3  2019      1026314  1850177000/1850182500/2910000502    고속도로  호남고속도로지선   
4  2019      1026318                        1850058200  특별광역시도      동서대로   

          권역  연장(km)  차선수  전체-평일_전일  전체-평일_1시  ...  트럭-평일_15시  트럭-평일_16시  \
0   대전광역시 서구  0.5991    1      4800        54  ...        219         74   
1   대전광역시 서구  0.6456    1      2375        30  ...         51         49   
2   대전광역시 서

In [13]:
# ============================================================
# STEP 1-5. 정거장 500m × 평일 추정교통량 매칭 (자동 컬럼 인식 버전)
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1) 추정교통량 CSV 불러오기
# ------------------------------------------------------------
traffic = pd.read_csv("C:/Users/dhso2/Downloads/17.대전광역시_평일_추정교통량.csv")

traffic = traffic.rename(columns={"5.5 LINK ID": "k_link_id"})

print("추정교통량 CSV 로드 완료")
print(traffic.head())


# ------------------------------------------------------------
# 2) '전일' 기준 컬럼 자동 탐색
# ------------------------------------------------------------
total_col   = [c for c in traffic.columns if "전체-평일_전일" in c][0]
car_col     = [c for c in traffic.columns if "승용"   in c and "_전일" in c][0]   # 승용차 컬럼 자동 탐색
bus_col     = [c for c in traffic.columns if "버스"   in c and "_전일" in c][0]
truck_col   = [c for c in traffic.columns if "트럭"   in c and "_전일" in c][0]

print("\n=== 자동 인식된 전일 기준 컬럼 ===")
print("전체 :", total_col)
print("승용 :", car_col)
print("버스 :", bus_col)
print("트럭 :", truck_col)


# ------------------------------------------------------------
# 3) Merge용 숫자 컬럼 추출
# ------------------------------------------------------------
numeric_cols = [col for col in traffic.columns if "평일" in col]
traffic_small = traffic[["k_link_id"] + numeric_cols]

# ------------------------------------------------------------
# 4) 정거장 500m buffer 내부 도로(join)와 merge
# ------------------------------------------------------------
join_traffic = join.merge(traffic_small, on="k_link_id", how="left")

# 숫자로 변환
for col in numeric_cols:
    join_traffic[col] = pd.to_numeric(join_traffic[col], errors='coerce')

# ------------------------------------------------------------
# 5) 정거장별 평균 추정교통량 계산
# ------------------------------------------------------------
station_traffic = join_traffic.groupby("정거장명")[numeric_cols].mean().reset_index()


# ------------------------------------------------------------
# 6) 전일 기준 Score 생성 (자동으로 탐색한 컬럼 사용)
# ------------------------------------------------------------
station_traffic["전체_전일_평균교통량"] = station_traffic[total_col]
station_traffic["승용_전일_평균교통량"] = station_traffic[car_col]
station_traffic["버스_전일_평균교통량"] = station_traffic[bus_col]
station_traffic["트럭_전일_평균교통량"] = station_traffic[truck_col]

traffic_summary = station_traffic[
    ["정거장명", 
     "전체_전일_평균교통량", 
     "승용_전일_평균교통량", 
     "버스_전일_평균교통량", 
     "트럭_전일_평균교통량"]
]

print("\n🚀 최종 정거장별 추정교통량 Summary:")
print(traffic_summary.sort_values("전체_전일_평균교통량", ascending=False))


추정교통량 CSV 로드 완료
   기준연도  k_link_id                       ITS LINK ID    도로등급       도로명  \
0  2019    1026307                        1850000300     연결로  호남고속도로지선   
1  2019    1026308                        1850000700     연결로  호남고속도로지선   
2  2019    1026309                        1850000600    고속도로  호남고속도로지선   
3  2019    1026314  1850177000/1850182500/2910000502    고속도로  호남고속도로지선   
4  2019    1026318                        1850058200  특별광역시도      동서대로   

          권역  연장(km)  차선수  전체-평일_전일  전체-평일_1시  ...  트럭-평일_15시  트럭-평일_16시  \
0   대전광역시 서구  0.5991    1      4800        54  ...        219         74   
1   대전광역시 서구  0.6456    1      2375        30  ...         51         49   
2   대전광역시 서구  0.4280    2     21918       173  ...        455        460   
3  대전광역시 유성구  5.9944    2     22524       256  ...        474        451   
4   대전광역시 서구  0.5269    4     16565       196  ...        132        141   

   트럭-평일_17시  트럭-평일_18시  트럭-평일_19시  트럭-평일_20시  트럭-평일_21시  트럭-평일_22시  \
0        10

## 🚊 STEP 1-6 완성 코드 (정거장 500m × 차량주행거리 매칭)

In [16]:
# ============================================================
# STEP 1-6. 정거장 500m Buffer × 차량주행거리(VKT) 매칭 (수정본)
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1) 차량주행거리 CSV 불러오기
# ------------------------------------------------------------
vkt = pd.read_csv("C:/Users/dhso2/Downloads/18.대전광역시_평일_차량주행거리.csv")

# 링크 ID 통일
vkt = vkt.rename(columns={"5.5 LINK ID": "k_link_id"})

print("차량주행거리 CSV 구조:")
print(vkt.head())


# ------------------------------------------------------------
# 2) 필요한 컬럼만 추출
# ------------------------------------------------------------
# 차량주행거리에서 숫자 컬럼은 '평일' 하나만 존재!
vkt_small = vkt[["k_link_id", "평일"]].copy()

# 숫자로 변환
vkt_small["평일"] = pd.to_numeric(vkt_small["평일"], errors='coerce')


# ------------------------------------------------------------
# 3) 정거장 buffer 내부 도로(join)와 merge
# ------------------------------------------------------------
join_vkt = join.merge(vkt_small, on="k_link_id", how="left")

print("\n정거장 내 도로 + 차량주행거리 merge 완료")
print(join_vkt.head())


# ------------------------------------------------------------
# 4) 정거장별 평균 차량주행거리 계산
# ------------------------------------------------------------
station_vkt = join_vkt.groupby("정거장명")["평일"].mean().reset_index()
station_vkt = station_vkt.rename(columns={"평일": "평일_평균_주행거리"})

print("\n🚦 정거장별 평균 차량주행거리(VKT):")
print(station_vkt.sort_values("평일_평균_주행거리", ascending=False))


차량주행거리 CSV 구조:
   기준연도  k_link_id                       ITS LINK ID    도로등급       도로명  \
0  2019    1026307                        1850000300     연결로  호남고속도로지선   
1  2019    1026308                        1850000700     연결로  호남고속도로지선   
2  2019    1026309                        1850000600    고속도로  호남고속도로지선   
3  2019    1026314  1850177000/1850182500/2910000502    고속도로  호남고속도로지선   
4  2019    1026318                        1850058200  특별광역시도      동서대로   

          권역  연장(km)  차선수      평일  
0   대전광역시 서구  0.5991    1    2875  
1   대전광역시 서구  0.6456    1    1533  
2   대전광역시 서구  0.4280    2    9381  
3  대전광역시 유성구  5.9944    2  135018  
4   대전광역시 서구  0.5269    4    8728  

정거장 내 도로 + 차량주행거리 merge 완료
   k_link_id  fnode_id  tnode_id road_name road_no  road_rank  link_type  \
0  1026322.0  278075.0  277811.0       배재로       0      104.0    32768.0   
1  1026322.0  278075.0  277811.0       배재로       0      104.0    32768.0   
2  1026322.0  278075.0  277811.0       배재로       0      104.0    327

## 🚊 STEP 1-7 — 5종 배출량 전체 자동 매핑 코드

In [21]:
# ============================================================
# STEP 1-7. 정거장 500m × 배출량 5종 전체 자동 매칭 (완전 수정본)
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1) 파일 목록 정의 (5종)
# ------------------------------------------------------------
pollution_files = {
    "CO":   "C:/Users/dhso2/Downloads/25.대전광역시_평일_일산화탄소 배출량.csv",
    "NOx":  "C:/Users/dhso2/Downloads/26.대전광역시_평일_질소산화물 배출량.csv",
    "PM":   "C:/Users/dhso2/Downloads/27.대전광역시_평일_미세먼지 배출량.csv",
    "VOC":  "C:/Users/dhso2/Downloads/28.대전광역시_평일_휘발성유기화합물 배출량.csv",
    "CO2":  "C:/Users/dhso2/Downloads/24.대전광역시_평일_이산화탄소 배출량.csv"
}

result_list = []


# ------------------------------------------------------------
# 2) 5종 배출량 처리
# ------------------------------------------------------------
for pol_name, file_path in pollution_files.items():
    print(f"\n▶ Processing {pol_name}: {file_path}")

    # CSV 로드
    df = pd.read_csv(file_path)

    # 링크 ID 통일
    df = df.rename(columns={"5.5 LINK ID": "k_link_id"})

    # 배출량 컬럼 4개 자동 선택
    pol_cols = ["전체", "승용차", "버스", "트럭"]

    df_small = df[["k_link_id"] + pol_cols]

    # 숫자로 변환
    for col in pol_cols:
        df_small[col] = pd.to_numeric(df_small[col], errors='coerce')

    # merge (정거장 buffer 내부 도로)
    join_pol = join.merge(df_small, on="k_link_id", how="left")

    # 정거장별 평균 배출량 계산
    station_pol = join_pol.groupby("정거장명")[pol_cols].mean().reset_index()

    # 컬럼명에 pollutant 이름 접두사 붙이기
    station_pol = station_pol.rename(columns={
        "전체":   f"{pol_name}_전체",
        "승용차": f"{pol_name}_승용차",
        "버스":   f"{pol_name}_버스",
        "트럭":   f"{pol_name}_트럭"
    })

    result_list.append(station_pol)


# ------------------------------------------------------------
# 3) 5종 배출량 통합
# ------------------------------------------------------------
pollution_final = result_list[0]

for df in result_list[1:]:
    pollution_final = pollution_final.merge(df, on="정거장명", how="left")


# ------------------------------------------------------------
# 4) 최종 출력
# ------------------------------------------------------------
print("\n===================================================")
print("🚀 정거장별 배출량 5종 Summary")
print("===================================================")
print(pollution_final.sort_values("CO2_전체", ascending=False))



▶ Processing CO: C:/Users/dhso2/Downloads/25.대전광역시_평일_일산화탄소 배출량.csv

▶ Processing NOx: C:/Users/dhso2/Downloads/26.대전광역시_평일_질소산화물 배출량.csv

▶ Processing PM: C:/Users/dhso2/Downloads/27.대전광역시_평일_미세먼지 배출량.csv

▶ Processing VOC: C:/Users/dhso2/Downloads/28.대전광역시_평일_휘발성유기화합물 배출량.csv

▶ Processing CO2: C:/Users/dhso2/Downloads/24.대전광역시_평일_이산화탄소 배출량.csv

🚀 정거장별 배출량 5종 Summary
       정거장명         CO_전체        CO_승용차        CO_버스        CO_트럭  \
24       연축  35850.343200  18220.498933  9676.311867  7953.532133   
32       읍내  11982.711026   6717.523692  2576.979179  2688.207974   
18       상대  18126.904500  13736.745125  2691.936000  1698.223250   
0       가수원  13378.209833   8992.062733  3030.016567  1356.130467   
11       동부  12574.166188   8731.141531  2567.266813  1275.757969   
15      목원대   2945.238500   1430.601500   583.380500   931.256500   
12  동부여성가족원   6730.933760   3939.391053  1377.888427  1413.654213   
29     유성온천  12834.134375   9392.070150  2541.250800   900.813475   
25    

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11432\202534786.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_small[col] = pd.to_numeric(df_small[col], errors='coerce')
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11432\202534786.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_small[col] = pd.to_numeric(df_small[col], errors='coerce')
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11432\202534786.py:40: SettingWithCopyWarning: 
A value is trying to be set on a c

# STEP 2

In [24]:
# ============================================================
# 혼잡강도 CSV 불러오기
# ============================================================

import pandas as pd

# 혼잡강도 파일 로드
cong = pd.read_csv("C:/Users/dhso2/Downloads/22.대전광역시_평일_혼잡강도.csv")

# 링크 ID 통일
cong = cong.rename(columns={"5.5 LINK ID": "k_link_id"})

# 필요한 컬럼만 추출
cong_small = cong[["k_link_id", "혼잡빈도강도", "혼잡시간강도", "혼잡기대강도"]].copy()

print("혼잡강도 데이터 확인")
print(cong_small.head())


# ============================================================
# 정거장 buffer × 혼잡강도 merge
# join → (도로 링크 + 정거장 buffer spatial join) 결과여야 함
# ============================================================

join_cong = join.merge(cong_small, on="k_link_id", how="left")

print("\n정거장 buffer 내부 도로 × 혼잡강도 merge 성공!")
print(join_cong.head())


# ============================================================
# 정거장별 평균 혼잡강도 계산
# ============================================================

station_cong = join_cong.groupby("정거장명")[["혼잡빈도강도", "혼잡시간강도", "혼잡기대강도"]].mean().reset_index()

print("\n🚦 정거장별 혼잡강도 요약:")
print(station_cong.head())


혼잡강도 데이터 확인
   k_link_id  혼잡빈도강도  혼잡시간강도  혼잡기대강도
0    1026309      19      25      30
1    1026314      29      36      29
2    1026318      24      40      62
3    1026319      31      54      60
4    1026320      27      47      65

정거장 buffer 내부 도로 × 혼잡강도 merge 성공!
   k_link_id  fnode_id  tnode_id road_name road_no  road_rank  link_type  \
0  1026322.0  278075.0  277811.0       배재로       0      104.0    32768.0   
1  1026322.0  278075.0  277811.0       배재로       0      104.0    32768.0   
2  1026322.0  278075.0  277811.0       배재로       0      104.0    32768.0   
3  1026322.0  278075.0  277811.0       배재로       0      104.0    32768.0   
4  1026323.0  277828.0  277911.0       구봉로       0      104.0    32768.0   

   lane road_info  sido_id  sigungu_id      emd_id  k_length  \
0   4.0      None  25000.0     25030.0  25030530.0     1.037   
1   4.0      None  25000.0     25030.0  25030530.0     1.037   
2   4.0      None  25000.0     25030.0  25030530.0     1.037   
3   4.0      None 

In [25]:
# ============================================================
# STEP 2. 정거장 종합 Score 계산
# ============================================================

import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# ------------------------------------------------------------
# 1) 지표 통합 (정거장 기준 outer merge)
# ------------------------------------------------------------
df_merged = (
    station_cong
    .merge(speed_summary, on="정거장명", how="outer")
    .merge(traffic_summary, on="정거장명", how="outer")
    .merge(station_vkt, on="정거장명", how="outer")
    .merge(pollution_final, on="정거장명", how="outer")
)

print("\n[STEP 2-1] 통합 테이블 생성 완료")
print(df_merged.head())


# ------------------------------------------------------------
# 2) 결측치 처리
# ------------------------------------------------------------
df_merged = df_merged.fillna(0)

print("\n[STEP 2-2] 결측치 처리 완료")


# ------------------------------------------------------------
# 3) 정규화 (MinMaxScaling)
# ------------------------------------------------------------
score_cols = [
    "혼잡빈도강도", "혼잡시간강도", "혼잡기대강도",
    "혼잡시_전일_평균속도", "정상시_전일_평균속도",
    "전체_전일_평균교통량", "승용_전일_평균교통량",
    "버스_전일_평균교통량", "트럭_전일_평균교통량",
    "평일_평균_주행거리",
    "CO_전체", "NOx_전체", "PM_전체", "VOC_전체", "CO2_전체"
]

scaler = MinMaxScaler()
df_merged_norm = df_merged.copy()
df_merged_norm[score_cols] = scaler.fit_transform(df_merged[score_cols])

print("\n[STEP 2-3] 정규화 완료")


# ------------------------------------------------------------
# 4) 종합 Score 계산
# ------------------------------------------------------------
df_merged_norm["종합점수"] = (
    0.30 * df_merged_norm[["혼잡빈도강도", "혼잡시간강도", "혼잡기대강도"]].mean(axis=1) +
    0.20 * df_merged_norm[["혼잡시_전일_평균속도", "정상시_전일_평균속도"]].mean(axis=1) +
    0.20 * df_merged_norm[["전체_전일_평균교통량", "승용_전일_평균교통량",
                           "버스_전일_평균교통량", "트럭_전일_평균교통량"]].mean(axis=1) +
    0.10 * df_merged_norm["평일_평균_주행거리"] +
    0.20 * df_merged_norm[["CO_전체", "NOx_전체", "PM_전체", "VOC_전체", "CO2_전체"]].mean(axis=1)
)


station_score = df_merged_norm[["정거장명", "종합점수"]].sort_values("종합점수", ascending=False)

print("\n====================================================")
print("🚀 STEP 2 완료: 정거장 종합 Score TOP 10")
print("====================================================")
print(station_score.head(10))



[STEP 2-1] 통합 테이블 생성 완료
     정거장명     혼잡빈도강도     혼잡시간강도     혼잡기대강도  혼잡시_전일_평균속도  정상시_전일_평균속도  \
0     가수원  65.433333  72.633333  27.866667    29.894737    67.833333   
1      가양  29.062500  46.875000  33.593750    15.687500    41.531250   
2      관저  67.714286  76.678571  29.375000    32.285714    67.267857   
3    관저 4  56.140187  68.551402  30.887850    25.197647    53.736471   
4  농수산물시장  36.347826  55.836957  31.326087    20.608696    45.815217   

    전체_전일_평균교통량   승용_전일_평균교통량  버스_전일_평균교통량  트럭_전일_평균교통량  ...     PM_버스  \
0  22047.100000  18162.133333   993.266667  2891.600000  ...  9.831067   
1  11751.093750   9487.843750   657.812500  1605.468750  ...  5.653344   
2  18593.196429  14963.107143   886.625000  2743.410714  ...  3.535393   
3  12673.250000  10007.925926   572.333333  2092.944444  ...  1.678794   
4  17961.217391  14637.750000   561.739130  2761.739130  ...  2.385043   

        PM_트럭       VOC_전체     VOC_승용차       VOC_버스      VOC_트럭        CO2_전체  \
0  107.514033  2

In [39]:
# ============================================================
# STEP 2. 정거장 종합 Score 계산 + CSV 저장
# ============================================================

import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# ------------------------------------------------------------
# 1) 지표 통합 (정거장 기준 outer merge)
# ------------------------------------------------------------
df_merged = (
    station_cong
    .merge(speed_summary, on="정거장명", how="outer")
    .merge(traffic_summary, on="정거장명", how="outer")
    .merge(station_vkt, on="정거장명", how="outer")
    .merge(pollution_final, on="정거장명", how="outer")
)

print("\n[STEP 2-1] 통합 테이블 생성 완료")
print(df_merged.head())

# 저장 ①: STEP2 원본 통합 테이블
df_merged.to_csv(
    "C:/Users/dhso2/Downloads/daejeon_tram_step2_merge_raw.csv",
    index=False, encoding="utf-8-sig"
)
print("📁 저장완료: step2_merge_raw.csv")

# ------------------------------------------------------------
# 2) 결측치 처리
# ------------------------------------------------------------
df_merged = df_merged.fillna(0)

print("\n[STEP 2-2] 결측치 처리 완료")

# ------------------------------------------------------------
# 3) 정규화 (MinMaxScaling)
# ------------------------------------------------------------
score_cols = [
    "혼잡빈도강도", "혼잡시간강도", "혼잡기대강도",
    "혼잡시_전일_평균속도", "정상시_전일_평균속도",
    "전체_전일_평균교통량", "승용_전일_평균교통량",
    "버스_전일_평균교통량", "트럭_전일_평균교통량",
    "평일_평균_주행거리",
    "CO_전체", "NOx_전체", "PM_전체", "VOC_전체", "CO2_전체"
]

scaler = MinMaxScaler()
df_merged_norm = df_merged.copy()
df_merged_norm[score_cols] = scaler.fit_transform(df_merged[score_cols])

print("\n[STEP 2-3] 정규화 완료")

# 저장 ②: 정규화된 통합 테이블
df_merged_norm.to_csv(
    "C:/Users/dhso2/Downloads/daejeon_tram_step2_merge_normalized.csv",
    index=False, encoding="utf-8-sig"
)
print("📁 저장완료: step2_merge_normalized.csv")

# ------------------------------------------------------------
# 4) 종합 Score 계산
# ------------------------------------------------------------
df_merged_norm["종합점수"] = (
    0.30 * df_merged_norm[["혼잡빈도강도", "혼잡시간강도", "혼잡기대강도"]].mean(axis=1) +
    0.20 * df_merged_norm[["혼잡시_전일_평균속도", "정상시_전일_평균속도"]].mean(axis=1) +
    0.20 * df_merged_norm[["전체_전일_평균교통량", "승용_전일_평균교통량",
                           "버스_전일_평균교통량", "트럭_전일_평균교통량"]].mean(axis=1) +
    0.10 * df_merged_norm["평일_평균_주행거리"] +
    0.20 * df_merged_norm[["CO_전체", "NOx_전체", "PM_전체", "VOC_전체", "CO2_전체"]].mean(axis=1)
)

station_score = df_merged_norm[["정거장명", "종합점수"]].sort_values("종합점수", ascending=False)

print("\n====================================================")
print("🚀 STEP 2 완료: 정거장 종합 Score TOP 10")
print("====================================================")
print(station_score.head(10))

# 저장 ③: 정거



[STEP 2-1] 통합 테이블 생성 완료
     정거장명     혼잡빈도강도     혼잡시간강도     혼잡기대강도  혼잡시_전일_평균속도  정상시_전일_평균속도  \
0     가수원  65.433333  72.633333  27.866667    29.894737    67.833333   
1      가양  29.062500  46.875000  33.593750    15.687500    41.531250   
2      관저  67.714286  76.678571  29.375000    32.285714    67.267857   
3    관저 4  56.140187  68.551402  30.887850    25.197647    53.736471   
4  농수산물시장  36.347826  55.836957  31.326087    20.608696    45.815217   

    전체_전일_평균교통량   승용_전일_평균교통량  버스_전일_평균교통량  트럭_전일_평균교통량  ...     PM_버스  \
0  22047.100000  18162.133333   993.266667  2891.600000  ...  9.831067   
1  11751.093750   9487.843750   657.812500  1605.468750  ...  5.653344   
2  18593.196429  14963.107143   886.625000  2743.410714  ...  3.535393   
3  12673.250000  10007.925926   572.333333  2092.944444  ...  1.678794   
4  17961.217391  14637.750000   561.739130  2761.739130  ...  2.385043   

        PM_트럭       VOC_전체     VOC_승용차       VOC_버스      VOC_트럭        CO2_전체  \
0  107.514033  2

# ⭐ STEP 3

## ⭐ STEP 3-A: 정거장 효과 구간 자동 분류 (우선 개선구간 / 제한구간)
우리가 이미 만든 station_score (정거장명 + 종합점수) 를 기반으로 다음 4단계로 자동 분류할 거야.

📌 분류 기준(권장)

- 상위 20% → 최우선 개선구간 (트램 효과 기대 MAX)

- 상위 20~50% → 우선 개선구간

- 하위 20~50% → 일반구간

- 하위 20% → 효과 제한구간

In [26]:
# ============================================================
# STEP 3-A. 정거장 효과 기대구간 vs 제한구간 자동 분류
# ============================================================

import numpy as np
import pandas as pd

df_score = station_score.copy()  # (정거장명, 종합점수)

# 순위 계산
df_score["rank"] = df_score["종합점수"].rank(ascending=False, method="dense")
df_score["percentile"] = df_score["rank"] / df_score["rank"].max()

# 구간 설정
def classify_zone(p):
    if p <= 0.20:
        return "① 최우선 개선구간 (Effect Max)"
    elif p <= 0.50:
        return "② 우선 개선구간"
    elif p <= 0.80:
        return "③ 일반 구간"
    else:
        return "④ 효과 제한구간"

df_score["구간"] = df_score["percentile"].apply(classify_zone)

print("\n====================================================")
print("🚀 STEP 3-A: 정거장 효과구간 분류 결과 (상위 10개)")
print("====================================================")
print(df_score.head(10))

# 전체 저장
effect_zone_table = df_score



🚀 STEP 3-A: 정거장 효과구간 분류 결과 (상위 10개)
     정거장명      종합점수  rank  percentile                       구간
24     연축  0.892728   1.0    0.023810  ① 최우선 개선구간 (Effect Max)
0     가수원  0.506303   2.0    0.047619  ① 최우선 개선구간 (Effect Max)
21   서대전역  0.487757   3.0    0.071429  ① 최우선 개선구간 (Effect Max)
17     복수  0.475715   4.0    0.095238  ① 최우선 개선구간 (Effect Max)
20  서대전 4  0.460662   5.0    0.119048  ① 최우선 개선구간 (Effect Max)
2      관저  0.447600   6.0    0.142857  ① 최우선 개선구간 (Effect Max)
38     진잠  0.444331   7.0    0.166667  ① 최우선 개선구간 (Effect Max)
32     읍내  0.438250   8.0    0.190476  ① 최우선 개선구간 (Effect Max)
29   유성온천  0.424717   9.0    0.214286                ② 우선 개선구간
33     인동  0.419428  10.0    0.238095                ② 우선 개선구간


## 🚀 STEP 3-B Folium 전체 코드

In [31]:
# ============================================================
# STEP 3-B. Folium 지도 시각화 (Threshold 오류 100% 해결 버전)
# ============================================================

import pandas as pd
import folium
from folium.plugins import MarkerCluster
import branca.colormap as cm

# 1) 정거장 좌표 CSV
stations = pd.read_csv("C:/Users/dhso2/Downloads/2호선_위_경도 (1).csv")

# 2) Score merge
map_df = stations.merge(effect_zone_table, on="정거장명", how="left")

# 3) 종합점수 정규화 (0~1)
map_df["norm_score"] = (
    map_df["종합점수"] - map_df["종합점수"].min()
) / (map_df["종합점수"].max() - map_df["종합점수"].min())


# ============================================================
# 🔥 Threshold 오류 없는 StepColormap 사용
# ============================================================

colormap = cm.StepColormap(
    colors=['blue', 'green', 'yellow', 'orange', 'red'],
    vmin=0.0,
    vmax=1.0,
    index=[0.0, 0.25, 0.50, 0.75, 1.0],
    caption="정거장 종합점수 (정규화)"
)


# 4) 지도 생성
m = folium.Map(
    location=[map_df["위도"].mean(), map_df["경도"].mean()],
    zoom_start=12,
    tiles="cartodbpositron"
)

cluster = MarkerCluster().add_to(m)


# 5) 마커 표시
for _, row in map_df.iterrows():
    score = row["norm_score"]
    raw = row["종합점수"]

    folium.CircleMarker(
        location=[row["위도"], row["경도"]],
        radius=8 + score * 10,         # 반경
        color=colormap(score),         # 안전한 StepColormap
        fill=True,
        fill_opacity=0.9,
        popup=folium.Popup(
            f"""
            <b>정거장명:</b> {row['정거장명']}<br>
            <b>정규화 점수:</b> {score:.4f}<br>
            <b>원본 점수:</b> {raw:.4f}<br>
            <b>구간:</b> {row['구간']}
            """,
            max_width=250
        )
    ).add_to(cluster)

colormap.add_to(m)


# 6) 저장
m.save("daejeon_tram_score_map.html")
print("\n🚀 지도 출력 완료 (StepColormap 적용): daejeon_tram_score_map.html")



🚀 지도 출력 완료 (StepColormap 적용): daejeon_tram_score_map.html


In [32]:
# ============================================================
# STEP 3-B. 정거장 종합점수 지도 시각화 (Folium) - 개선 버전
# ============================================================

import pandas as pd
import folium
from folium.plugins import MarkerCluster
import branca.colormap as cm

# 1) 정거장 좌표 CSV 불러오기
stations = pd.read_csv("C:/Users/dhso2/Downloads/2호선_위_경도 (1).csv")

# 2) Score 테이블과 merge
map_df = stations.merge(effect_zone_table, on="정거장명", how="left")

# (중요) 정거장 점수 없는 행 제거
map_df = map_df.dropna(subset=["종합점수"])

# 정규화 점수 만들기
min_s = map_df["종합점수"].min()
max_s = map_df["종합점수"].max()
map_df["norm_score"] = (map_df["종합점수"] - min_s) / (max_s - min_s)

# 3) Folium 지도 생성
center_lat = map_df["위도"].mean()
center_lon = map_df["경도"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="cartodbpositron")

# 4) 안정적인 LinearColormap (0~1)
colormap = cm.LinearColormap(
    colors=["blue", "green", "yellow", "orange", "red"],
    vmin=0, vmax=1,
    caption="정거장 종합점수 (정규화)"
)

# 5) 클러스터
cluster = MarkerCluster().add_to(m)

# 6) 정거장 표시
for _, row in map_df.iterrows():

    score = float(row["norm_score"])
    score = max(0, min(score, 1))   # clip

    folium.CircleMarker(
        location=[row["위도"], row["경도"]],
        radius=8 + score * 10,
        color=colormap(score),
        fill=True,
        fill_opacity=0.9,
        popup=folium.Popup(
            f"""
            <b>정거장명:</b> {row['정거장명']}<br>
            <b>정규화 점수(norm):</b> {score:.4f}<br>
            <b>원본 점수(raw):</b> {row['종합점수']:.4f}<br>
            <b>구간분류:</b> {row['구간']}
            """,
            max_width=250
        )
    ).add_to(cluster)

colormap.add_to(m)

# 7) 저장
m.save("daejeon_tram_score_map_fixed.html")
print("\n🚀 지도 저장 완료: daejeon_tram_score_map_fixed.html")



🚀 지도 저장 완료: daejeon_tram_score_map_fixed.html


In [33]:
# ============================================================
# STEP 3-B 전체 정거장 항상 표시 버전 (MarkerCluster 제거)
# ============================================================

import pandas as pd
import folium
import branca.colormap as cm

# 1) 정거장 좌표
stations = pd.read_csv("C:/Users/dhso2/Downloads/2호선_위_경도 (1).csv")

# 2) Score merge
map_df = stations.merge(effect_zone_table, on="정거장명", how="left")

# 정거장 점수 없는 데이터 제거
map_df = map_df.dropna(subset=["종합점수"])

# 종합점수 정규화
min_s = map_df["종합점수"].min()
max_s = map_df["종합점수"].max()
map_df["norm_score"] = (map_df["종합점수"] - min_s) / (max_s - min_s)

# 3) 지도 생성
m = folium.Map(
    location=[map_df["위도"].mean(), map_df["경도"].mean()],
    zoom_start=12,
    tiles="cartodbpositron"
)

# 안전한 colormap
colormap = cm.StepColormap(
    colors=['blue', 'green', 'yellow', 'orange', 'red'],
    index=[0, 0.25, 0.5, 0.75, 1],
    vmin=0, vmax=1,
    caption="정거장 종합점수 (정규화)"
)

# 4) 모든 정거장 표시 (Cluster 제거)
for _, row in map_df.iterrows():

    score = float(row["norm_score"])
    score = max(0, min(score, 1))

    folium.CircleMarker(
        location=[row["위도"], row["경도"]],
        radius=7 + score * 8,
        color=colormap(score),
        fill=True,
        fill_opacity=0.9,
        popup=folium.Popup(
            f"""
            <b>정거장명:</b> {row['정거장명']}<br>
            <b>점수(norm):</b> {score:.3f}<br>
            <b>점수(raw):</b> {row['종합점수']:.3f}<br>
            <b>구간:</b> {row['구간']}
            """,
            max_width=250
        )
    ).add_to(m)

colormap.add_to(m)

# 저장
m.save("daejeon_tram_score_map_all_visible.html")
print("🚀 모든 정거장 항상 보이는 지도 저장 완료!")


🚀 모든 정거장 항상 보이는 지도 저장 완료!


In [34]:
# ============================================================
# STEP 3-B. 모든 정거장 항상 보이는 시각화 (MarkerCluster 제거)
# ============================================================

import pandas as pd
import folium
import branca.colormap as cm

# 1) 정거장 좌표 CSV 불러오기
stations = pd.read_csv("C:/Users/dhso2/Downloads/2호선_위_경도 (1).csv")

# 2) Score 테이블과 merge
map_df = stations.merge(effect_zone_table, on="정거장명", how="left")

print("\n[지도 DataFrame 확인]")
print(map_df.head())

# (중요) 종합점수 없는 정거장은 제거
map_df = map_df.dropna(subset=["종합점수"])

# ---------------------------
# 🔥 종합점수 정규화
# ---------------------------
min_s = map_df["종합점수"].min()
max_s = map_df["종합점수"].max()
map_df["norm_score"] = (map_df["종합점수"] - min_s) / (max_s - min_s)


# ---------------------------
# 🔥 StepColormap 사용 (threshold 오류 없음)
# ---------------------------
colormap = cm.StepColormap(
    colors=["blue", "green", "yellow", "orange", "red"],
    index=[0.0, 0.25, 0.50, 0.75, 1.0],
    vmin=0.0,
    vmax=1.0,
    caption="정거장 종합점수 (정규화)"
)

# 3) Folium 지도 생성
m = folium.Map(
    location=[map_df["위도"].mean(), map_df["경도"].mean()],
    zoom_start=12,
    tiles="cartodbpositron"
)

# ---------------------------
# 🔥 모든 정거장 CircleMarker만 표시
# ---------------------------
for _, row in map_df.iterrows():
    score = float(row["norm_score"])
    score = max(0, min(score, 1))   # 안전 clip

    folium.CircleMarker(
        location=[row["위도"], row["경도"]],
        radius=7 + score * 8,   # 점수 기반 변화
        color=colormap(score),
        fill=True,
        fill_opacity=0.9,
        popup=folium.Popup(
            f"""
            <b>정거장명:</b> {row['정거장명']}<br>
            <b>정규화 점수(norm):</b> {score:.3f}<br>
            <b>원본 점수(raw):</b> {row['종합점수']:.3f}<br>
            <b>구간:</b> {row['구간']}
            """,
            max_width=300
        )
    ).add_to(m)

# 컬러맵 추가
colormap.add_to(m)

# 7) 파일 저장
m.save("daejeon_tram_score_map_all_visible.html")
print("\n🚀 모든 정거장이 항상 보이는 지도 저장 완료!")



[지도 DataFrame 확인]
   정거장ID   정거장명         위도          경도            _PARCEL_AD  \
0    201   서대전역  36.320142  127.404468      대전광역시 중구 오류동 195   
1    202  서대전 4  36.322083  127.411119      대전광역시 중구 오류동 195   
2    203     대사  36.319332  127.416331  대전광역시 중구 대사동 248-225   
3    204     대흥  36.318118  127.428438    대전광역시 중구 대사동 77-51   
4    205     인동  36.322842  127.437252       대전광역시 동구 인동 352   

             _ROAD_AD      종합점수  rank  percentile                       구간  
0   대전광역시 중구 계백로 1663  0.487757   3.0    0.071429  ① 최우선 개선구간 (Effect Max)  
1   대전광역시 중구 계백로 1719  0.460662   5.0    0.119048  ① 최우선 개선구간 (Effect Max)  
2    대전광역시 중구 계룡로 937  0.374156  14.0    0.333333                ② 우선 개선구간  
3  대전광역시 중구 충무로 104-1  0.319419  20.0    0.476190                ② 우선 개선구간  
4    대전광역시 동구 대전로 702  0.419428  10.0    0.238095                ② 우선 개선구간  

🚀 모든 정거장이 항상 보이는 지도 저장 완료!


In [35]:
# ============================================================
# STEP 3-C. 정거장 라벨 + 트램 노선 Polyline + Score 기반 스타일링
# ============================================================

import pandas as pd
import folium
import geopandas as gpd
import branca.colormap as cm

# ---------------------------
# 1) 필요한 데이터 불러오기
# ---------------------------

# 정거장 좌표
stations = pd.read_csv("C:/Users/dhso2/Downloads/2호선_위_경도 (1).csv")

# Score merge
map_df = stations.merge(effect_zone_table, on="정거장명", how="left")
map_df = map_df.dropna(subset=["종합점수"])

# 종합점수 정규화 (0~1)
min_s = map_df["종합점수"].min()
max_s = map_df["종합점수"].max()
map_df["norm_score"] = (map_df["종합점수"] - min_s) / (max_s - min_s)

# 트램 노선 (2022 도로망 기반)
road_2022 = gpd.read_file("C:/Users/dhso2/Downloads/32.대전광역시_level5_5_link_주요도로망_형상_2022.geojson")
road_2022 = road_2022.to_crs(4326)  # Folium은 반드시 WGS84 사용


# ---------------------------
# 2) 지도 생성
# ---------------------------
m = folium.Map(
    location=[map_df["위도"].mean(), map_df["경도"].mean()],
    zoom_start=12,
    tiles="cartodbpositron"
)

# ColorMap
colormap = cm.StepColormap(
    colors=["blue", "green", "yellow", "orange", "red"],
    index=[0, 0.25, 0.5, 0.75, 1],
    vmin=0,
    vmax=1,
    caption="정거장 종합점수 (정규화)"
)


# ---------------------------
# 3) 트램 노선 Polyline 그리기
# ---------------------------
for _, row in road_2022.iterrows():
    coords = list(row["geometry"].coords)
    folium.PolyLine(
        coords,
        weight=3,
        color="#444444",
        opacity=0.8
    ).add_to(m)


# ---------------------------
# 4) 정거장 CircleMarker + 라벨 추가
# ---------------------------
for _, row in map_df.iterrows():
    
    score = float(row["norm_score"])
    score = max(0, min(score, 1))   # 안전 clip

    # Circle marker
    folium.CircleMarker(
        location=[row["위도"], row["경도"]],
        radius=6 + score * 10,
        color=colormap(score),
        fill=True,
        fill_opacity=0.9
    ).add_to(m)

    # ---------------------------
    # 💬 텍스트 라벨 (Score 기반 스타일)
    # ---------------------------

    label_html = f"""
        <div style="
            font-size:{11 + score*8}px;
            color:{colormap(score)};
            font-weight:bold;
            text-shadow: -1px -1px 3px white, 1px -1px 3px white,
                         -1px 1px 3px white, 1px 1px 3px white;
            ">
            {row['정거장명']}
        </div>
    """

    folium.Marker(
        location=[row["위도"], row["경도"]],
        icon=folium.DivIcon(html=label_html)
    ).add_to(m)


# 컬러맵 추가
colormap.add_to(m)


# ---------------------------
# 5) 저장
# ---------------------------
m.save("daejeon_tram_score_map_full_visual.html")

print("\n🚀 최종 시각화 지도 저장 완료: daejeon_tram_score_map_full_visual.html")



🚀 최종 시각화 지도 저장 완료: daejeon_tram_score_map_full_visual.html


In [36]:
# ============================================================
# STEP 3-C. 정거장 + Polyline + Score 라벨 최종본
# ============================================================

import pandas as pd
import folium
import geopandas as gpd
import branca.colormap as cm

# ---------------------------
# 1) 데이터 불러오기
# ---------------------------

# 정거장 좌표
stations = pd.read_csv("C:/Users/dhso2/Downloads/2호선_위_경도 (1).csv")

# Score merge
map_df = stations.merge(effect_zone_table, on="정거장명", how="left")
map_df = map_df.dropna(subset=["종합점수"])

# 정규화
min_s = map_df["종합점수"].min()
max_s = map_df["종합점수"].max()
map_df["norm_score"] = (map_df["종합점수"] - min_s) / (max_s - min_s)

# 2호선 도로망
road_2022 = gpd.read_file("C:/Users/dhso2/Downloads/32.대전광역시_level5_5_link_주요도로망_형상_2022.geojson")

# 좌표계 변환 (5179 → 4326)
if road_2022.crs.to_epsg() != 4326:
    road_2022 = road_2022.to_crs(4326)


# ---------------------------
# 2) 지도 생성
# ---------------------------
m = folium.Map(
    location=[map_df["위도"].mean(), map_df["경도"].mean()],
    zoom_start=12,
    tiles="cartodbpositron"
)

# 색상 맵
colormap = cm.StepColormap(
    colors=["blue", "green", "yellow", "orange", "red"],
    index=[0, 0.25, 0.5, 0.75, 1],
    vmin=0,
    vmax=1,
    caption="정거장 종합점수 (정규화)"
)


# ---------------------------
# 3) 트램 2호선 PolyLine 시각화
# ---------------------------

# 도로명 기준으로 트램 노선 추출
tram_lines = road_2022[ road_2022["road_name"].isin(map_df["_ROAD_AD"].str.split().str[-1].unique()) ]

# 만약 이 방식으로 안 잡히면 전체 GeoJSON 라인만 표시
if len(tram_lines) == 0:
    tram_lines = road_2022

for _, row in tram_lines.iterrows():
    folium.PolyLine(
        locations=[(lat, lon) for lon, lat in row["geometry"].coords],  # (lat, lon)
        color="#333333",
        weight=4,
        opacity=0.8
    ).add_to(m)


# ---------------------------
# 4) 정거장 CircleMarker + 라벨
# ---------------------------
for _, row in map_df.iterrows():

    score = float(row["norm_score"])

    # Marker
    folium.CircleMarker(
        location=[row["위도"], row["경도"]],
        radius=7 + score * 8,
        color=colormap(score),
        fill=True,
        fill_opacity=0.9
    ).add_to(m)

    # 라벨
    label_html = f"""
        <div style="
            font-size:{11 + score*8}px;
            color:{colormap(score)};
            font-weight:bold;
            text-shadow: -1px -1px 3px white, 1px -1px 3px white,
                         -1px 1px 3px white, 1px 1px 3px white;
        ">
            {row['정거장명']}
        </div>
    """

    folium.Marker(
        location=[row["위도"], row["경도"]],
        icon=folium.DivIcon(html=label_html)
    ).add_to(m)


colormap.add_to(m)

m.save("daejeon_tram_score_map_full_visual_with_line.html")

print("🚀 폴리라인 포함 완성! daejeon_tram_score_map_full_visual_with_line.html")


🚀 폴리라인 포함 완성! daejeon_tram_score_map_full_visual_with_line.html


In [37]:
# ============================================================
# 📌 최종 전처리된 분석용 DataFrame 저장
# ============================================================

output_path = "C:/Users/dhso2/Downloads/daejeon_tram_final_table.csv"

effect_zone_table.to_csv(output_path, index=False, encoding="utf-8-sig")

print("✅ 저장 완료 :", output_path)
print(effect_zone_table.head())


✅ 저장 완료 : C:/Users/dhso2/Downloads/daejeon_tram_final_table.csv
     정거장명      종합점수  rank  percentile                       구간
24     연축  0.892728   1.0    0.023810  ① 최우선 개선구간 (Effect Max)
0     가수원  0.506303   2.0    0.047619  ① 최우선 개선구간 (Effect Max)
21   서대전역  0.487757   3.0    0.071429  ① 최우선 개선구간 (Effect Max)
17     복수  0.475715   4.0    0.095238  ① 최우선 개선구간 (Effect Max)
20  서대전 4  0.460662   5.0    0.119048  ① 최우선 개선구간 (Effect Max)
